# 07 ML Benchmark Models

This notebook screens machine-learning model families for weather-driven demand forecasting. It fits benchmark models with fixed starting configurations across the feature sets defined by the notebook 06 registry, providing an initial comparison before LightGBM tuning and interpretability analysis.

The notebook is a model-family screening step, not the final tuning or rolling-origin evaluation notebook.


## Pipeline position

This notebook follows `notebooks/06_build_ml_panel.ipynb` and precedes LightGBM tuning in notebook 08. It reads the ML panel and feature registry, fits benchmark model families, and writes screening outputs under `outputs/ml_models/`.


## Inputs and outputs

Inputs are `data/processed/ml_forecast_panel_full.parquet` and `outputs/ml_panel/ml_panel_feature_registry.csv`. Output filenames include `{RUN_MODE}`:

- `outputs/ml_models/benchmark_metrics_{run_mode}.csv`
- `outputs/ml_models/benchmark_predictions_{run_mode}.parquet`
- `outputs/ml_models/benchmark_model_summary_{run_mode}.csv`
- `outputs/ml_models/feature_set_summary_{run_mode}.csv`
- `outputs/ml_models/benchmark_gain_summary_{run_mode}.csv`


## Model specifications

Feature sets are constructed from the registry flags to stay consistent with notebook 06. M1 is the no-weather baseline with calendar, store/category, opening-status, historical sales, and origin-safe campaign history. M2 adds operational point forecast weather and origin-safe historical observed weather, but no realised target-day weather or uncertainty/probability features. M3 adds realised target-day weather as a non-operational perfect-information benchmark. M4 adds synthetic forecast uncertainty, interval, and probability features to M2, while still excluding realised target-day weather.

`AvdelingID` and `Analyse_Kategori` are intentionally added back as categorical features even though they sit in the registry `key` group. `origin_season` is diagnostic-only and excluded. Realised target-day weather columns and raw target-date campaign columns are flagged with `leakage_risk = True`; validation below prevents them from entering M1, M2, or M4.


## Evaluation design and run modes

Notebook 07 uses a forecast-origin-aware fixed chronological holdout for efficient model-family and feature-set screening. It is not the final rolling-origin evaluation. Train rows satisfy `target_date <= TRAIN_END_DATE`. Test rows satisfy `origin_date >= TEST_START_DATE` and `target_date <= TEST_END_DATE`. The test set is defined by forecast origins rather than target dates so that, for every horizon, test forecasts are not constructed from origins inside the training period. Rows in the gap between the training target cutoff and the test-origin start are intentionally excluded.

`RUN_MODE = "quick"` uses a small horizon/model subset for fast validation. `RUN_MODE = "full"` uses the full benchmark configuration for thesis-level screening outputs. Final expanding-window rolling-origin evaluation is out of scope for this notebook.


## Environment and portable paths

The setup cell defines project-root discovery, output folders, run mode paths, and helper functions for benchmark execution.


In [ ]:
import gc
import time
import warnings
from html import escape as _mfs_html_escape
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

MARKER_FILES = ("README_FOR_CODEX.md", "AGENTS.md", "pyproject.toml")


def find_project_root(start=None):
    """Walk upward until a project marker file is found."""
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if any((candidate / marker).exists() for marker in MARKER_FILES):
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Start Jupyter from inside the project "
        "folder or make sure README_FOR_CODEX.md, AGENTS.md, or pyproject.toml exists."
    )


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_ML_PANEL_DIR = OUTPUT_DIR / "ml_panel"
OUTPUT_ML_MODELS_DIR = OUTPUT_DIR / "ml_models"

ML_PANEL_PATH = PROCESSED_DIR / "ml_forecast_panel_full.parquet"
FEATURE_REGISTRY_PATH = OUTPUT_ML_PANEL_DIR / "ml_panel_feature_registry.csv"

OUTPUT_ML_MODELS_DIR.mkdir(parents=True, exist_ok=True)


def project_relative(path):
    path = Path(path)
    try:
        return path.relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        return str(path)


def require_file(path, description):
    if not Path(path).exists():
        raise FileNotFoundError(
            f"Missing {description}: {project_relative(path)}. "
            "Rerun the producing notebook before running this notebook."
        )


print(f"Project root: {PROJECT_ROOT}")
print(f"ML panel input: {project_relative(ML_PANEL_PATH)}")
print(f"Feature registry input: {project_relative(FEATURE_REGISTRY_PATH)}")
print(f"Model outputs folder: {project_relative(OUTPUT_ML_MODELS_DIR)}")


## Imports and package availability

The notebook detects which model packages are installed. Unavailable families are skipped with a recorded reason rather than failing the benchmark.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

MODEL_AVAILABILITY = {"random_forest": (True, "")}

try:
    from xgboost import XGBRegressor  # noqa: F401
    MODEL_AVAILABILITY["xgboost"] = (True, "")
except Exception as exc:  # pragma: no cover
    MODEL_AVAILABILITY["xgboost"] = (False, f"import_failed: {type(exc).__name__}: {exc}")

try:
    from lightgbm import LGBMRegressor  # noqa: F401
    MODEL_AVAILABILITY["lightgbm"] = (True, "")
except Exception as exc:  # pragma: no cover
    MODEL_AVAILABILITY["lightgbm"] = (False, f"import_failed: {type(exc).__name__}: {exc}")

try:
    from catboost import CatBoostRegressor  # noqa: F401
    # CatBoost <= 1.2.8 lacks __sklearn_tags__;
    # delegate to BaseEstimator for sklearn pipeline compatibility.
    from sklearn.base import BaseEstimator as _BaseEstimator
    if not hasattr(CatBoostRegressor, "__sklearn_tags__"):

        def _catboost_sklearn_tags(self):
            return _BaseEstimator.__sklearn_tags__(self)

        CatBoostRegressor.__sklearn_tags__ = _catboost_sklearn_tags
    MODEL_AVAILABILITY["catboost"] = (True, "")
except Exception as exc:  # pragma: no cover
    MODEL_AVAILABILITY["catboost"] = (False, f"import_failed: {type(exc).__name__}: {exc}")

print("Model package availability:")
for family, (available, reason) in MODEL_AVAILABILITY.items():
    print(f"  {family}: available={available} {reason}")


## Configuration

This cell defines the chronological holdout, run mode, horizons, model families, prediction clipping, future stubs, and output filenames. Quick mode is for debugging; full mode is for thesis-level screening outputs.


In [ ]:
RUN_MODE = "full"  # change to "full" for thesis-level benchmark outputs.
assert RUN_MODE in {"quick", "full"}, f"Unknown RUN_MODE: {RUN_MODE!r}"

# Forecast-origin-aware holdout: train target_date <= TRAIN_END_DATE;
# test origin_date >= TEST_START_DATE and target_date <= TEST_END_DATE.
# Gap rows between the training target cutoff and test-origin start are intentionally excluded.
TRAIN_END_DATE = pd.Timestamp("2024-07-31")
TEST_START_DATE = pd.Timestamp("2024-08-01")
TEST_END_DATE = pd.Timestamp("2025-07-31")

# Clip negative predictions because Antall_total is non-negative.
CLIP_NEGATIVE_PREDICTIONS = True

# RUN_MODE controls horizons, model families, and estimator counts.
# Horizon tiers: h=0-h=2 are archived MEPS evidence;
# h=3-h=10 are synthetic emulator scenario evidence.
# h=0 is included by default as a same-day/nowcast-like horizon.
INCLUDE_H0 = True

if RUN_MODE == "quick":
    HORIZONS_TO_RUN = [0, 1, 2, 3, 7, 10]
    MODEL_FAMILIES = ["random_forest", "xgboost"]
else:
    HORIZONS_TO_RUN = list(range(0, 11))
    MODEL_FAMILIES = ["random_forest", "xgboost", "lightgbm", "catboost"]

if not INCLUDE_H0:
    HORIZONS_TO_RUN = [h for h in HORIZONS_TO_RUN if h != 0]

FEATURE_SETS_TO_RUN = ["M1", "M2", "M3", "M4"]
RANDOM_STATE = 42

# Disabled placeholder for future rolling-origin evaluation definitions.
ROLLING_ORIGIN_FOLDS = []  # noqa: F841

# Disabled placeholder for future M4 ablation studies;
# standard benchmark behaviour is unchanged while False.
ENABLE_M4_ABLATIONS = False
M4_ABLATION_SETS_TODO = {
    "M4_std_only": "M2 + *_fcst_std",
    "M4_interval_only": "M2 + *_fcst_p10 + *_fcst_p90 + *_fcst_interval_width",
    "M4_precip_uncertainty": "M2 + precip wet probability + precip amount uncertainty",
    "M4_cloud_probability": "M2 + cloud regime probabilities",
    "M4_full": "current M4 (active default)",
}
assert ENABLE_M4_ABLATIONS is False, (
    "ENABLE_M4_ABLATIONS must remain False until ablation feature sets are wired "
    "into the benchmark loop. See M4_ABLATION_SETS_TODO."
)


def horizon_tier_for(h):
    """Map a horizon to its evidence-tier label used in audit outputs."""
    h_int = int(h)
    if h_int in (0, 1, 2):
        return "actual_meps_h0_h2"
    if h_int in (3, 4, 5, 6, 7, 8, 9, 10):
        return "synthetic_scenario_h3_h10"
    return f"unsupported_h{h_int}"


def output_path(suffix):
    return OUTPUT_ML_MODELS_DIR / f"{suffix}_{RUN_MODE}{'.parquet' if suffix.endswith('predictions') else '.csv'}"


METRICS_PATH = output_path("benchmark_metrics")
PREDICTIONS_PATH = output_path("benchmark_predictions")
MODEL_SUMMARY_PATH = output_path("benchmark_model_summary")
FEATURE_SET_SUMMARY_PATH = output_path("feature_set_summary")
GAIN_SUMMARY_PATH = output_path("benchmark_gain_summary")
WEATHER_VALUE_DECOMPOSITION_PATH = output_path("benchmark_weather_value_decomposition")
M4_VS_M2_DIAGNOSTIC_PATH = output_path("benchmark_m4_vs_m2_diagnostic")
M4_OVERFIT_RISK_PATH = output_path("benchmark_m4_overfitting_risk_summary")

print(f"RUN_MODE: {RUN_MODE}")
print(f"HORIZONS_TO_RUN: {HORIZONS_TO_RUN}")
print(f"MODEL_FAMILIES: {MODEL_FAMILIES}")
print(f"FEATURE_SETS_TO_RUN: {FEATURE_SETS_TO_RUN}")
print(f"Train: target_date <= {TRAIN_END_DATE.date()}")
print(f"Test:  origin_date >= {TEST_START_DATE.date()} AND target_date <= {TEST_END_DATE.date()}")
print()
print(f"Metrics output: {project_relative(METRICS_PATH)}")
print(f"Predictions output: {project_relative(PREDICTIONS_PATH)}")
print(f"Model summary output: {project_relative(MODEL_SUMMARY_PATH)}")
print(f"Feature set summary output: {project_relative(FEATURE_SET_SUMMARY_PATH)}")
print(f"Gain summary output: {project_relative(GAIN_SUMMARY_PATH)}")


## Feature-set construction

The feature sets are derived from registry flags. `AvdelingID` and `Analyse_Kategori` are explicitly included as categorical features because they encode store and category identity. Other key columns, target columns, and `origin_season` are excluded.


In [ ]:
require_file(FEATURE_REGISTRY_PATH, "feature registry from notebook 06")
require_file(ML_PANEL_PATH, "ML forecast panel from notebook 06")

feature_registry = pd.read_csv(FEATURE_REGISTRY_PATH)


def _to_bool(series):
    return series.astype(str).str.lower().isin(["true", "1", "yes"])


for column in [
    "allowed_m1_baseline",
    "allowed_m2_synthetic_point_weather",
    "allowed_m3_perfect_information",
    "allowed_m4_synthetic_engineered_weather",
    "leakage_risk",
    "known_at_origin",
]:
    feature_registry[column] = _to_bool(feature_registry[column])

KEY_COLUMNS = feature_registry.loc[feature_registry["role"] == "key", "column"].tolist()
TARGET_COLUMNS = feature_registry.loc[
    feature_registry["role"].isin(["target", "robustness_target"]),
    "column",
].tolist()

# AvdelingID and Analyse_Kategori are key columns but intentionally used as categorical identifiers.
CATEGORICAL_FEATURE_CANDIDATES = [
    "AvdelingID",
    "AvdelingTekst",
    "Region",
    "Analyse_Kategori",
    "Ukedag",
    "HelligdagNavn",
    "season",
]


FORBIDDEN_FEATURE_COLUMNS = set(KEY_COLUMNS + TARGET_COLUMNS + ["origin_season"]) - {
    "AvdelingID",
    "Analyse_Kategori",
}


def feature_set_columns(model_label):
    flag = {
        "M1": "allowed_m1_baseline",
        "M2": "allowed_m2_synthetic_point_weather",
        "M3": "allowed_m3_perfect_information",
        "M4": "allowed_m4_synthetic_engineered_weather",
    }[model_label]
    registry_features = feature_registry.loc[feature_registry[flag], "column"].tolist()
    # Exclude targets, non-exception key columns, and origin_season.
    registry_features = [
        c for c in registry_features if c not in FORBIDDEN_FEATURE_COLUMNS
    ]
    # Add store/category identifiers used by every model.
    explicit_categoricals = [
        c for c in CATEGORICAL_FEATURE_CANDIDATES if c in {"AvdelingID", "Analyse_Kategori"}
    ]
    feature_columns = list(dict.fromkeys(explicit_categoricals + registry_features))
    return feature_columns


feature_sets = {label: feature_set_columns(label) for label in FEATURE_SETS_TO_RUN}

for label, columns in feature_sets.items():
    print(f"{label}: {len(columns)} features")

# Mark selected categorical features for one-hot encoding.

def split_numeric_categorical(columns):
    categorical = [c for c in columns if c in CATEGORICAL_FEATURE_CANDIDATES]
    numeric = [c for c in columns if c not in CATEGORICAL_FEATURE_CANDIDATES]
    return numeric, categorical


feature_set_numeric = {}
feature_set_categorical = {}
for label, columns in feature_sets.items():
    num, cat = split_numeric_categorical(columns)
    feature_set_numeric[label] = num
    feature_set_categorical[label] = cat
    print(f"{label}: {len(num)} numeric, {len(cat)} categorical")


## Pre-fit validation checks

Before fitting, the notebook verifies that feature sets respect the operational leakage boundary. M1, M2, and M4 must not contain `leakage_risk = True` columns. M3 may include realised target-day weather but no other leakage-risk columns. The checks also exclude `origin_season`, raw target-date campaign columns, target columns, and features missing from the panel schema.


In [ ]:
panel_columns = list(pq.ParquetFile(ML_PANEL_PATH).schema_arrow.names)

leakage_columns = set(feature_registry.loc[feature_registry["leakage_risk"], "column"])
realised_weather_columns = set(
    feature_registry.loc[feature_registry["feature_group"] == "realised_weather_perfect_information", "column"]
)
raw_campaign_columns = set(
    feature_registry.loc[feature_registry["feature_group"] == "raw_campaign_diagnostic", "column"]
)


def assert_no_leakage(model_label):
    fset = set(feature_sets[model_label])
    if model_label in {"M1", "M2", "M4"}:
        violations = fset & leakage_columns
        if violations:
            raise AssertionError(f"{model_label} contains leakage_risk columns: {sorted(violations)}")
    elif model_label == "M3":
        # M3 may include realised-weather columns, but no other leakage-risk column.
        violations = (fset & leakage_columns) - realised_weather_columns
        if violations:
            raise AssertionError(f"M3 contains non-realised-weather leakage_risk columns: {sorted(violations)}")
    else:
        raise ValueError(f"Unknown model label: {model_label}")


def assert_no_forbidden_columns(model_label):
    fset = set(feature_sets[model_label])
    if "origin_season" in fset:
        raise AssertionError(
            f"{model_label} contains origin_season; it must be diagnostic only."
        )
    bad = fset & raw_campaign_columns
    if bad:
        raise AssertionError(f"{model_label} contains raw target-date campaign columns: {sorted(bad)}")
    bad = fset & set(TARGET_COLUMNS)
    if bad:
        raise AssertionError(f"{model_label} contains target columns: {sorted(bad)}")
    bad = fset & {"target_date", "origin_date", "horizon"}
    if bad:
        raise AssertionError(f"{model_label} contains non-categorical key columns: {sorted(bad)}")


def assert_features_exist(model_label):
    missing = [c for c in feature_sets[model_label] if c not in panel_columns]
    if missing:
        raise AssertionError(f"{model_label} references columns missing from the panel: {missing}")


for label in FEATURE_SETS_TO_RUN:
    assert_no_leakage(label)
    assert_no_forbidden_columns(label)
    assert_features_exist(label)

print("All pre-fit validation checks passed.")


## Load selected ML panel columns

The panel read is column-restricted to selected features, keys, and the target. Rows are filtered to `HORIZONS_TO_RUN` and to the forecast-origin-aware train/test window.


In [ ]:
selected_columns = sorted(
    set(
        KEY_COLUMNS
        + ["Antall_total"]
        + [c for label in FEATURE_SETS_TO_RUN for c in feature_sets[label]]
    )
)

panel = pd.read_parquet(ML_PANEL_PATH, columns=selected_columns)
panel["target_date"] = pd.to_datetime(panel["target_date"]).dt.normalize()
panel["origin_date"] = pd.to_datetime(panel["origin_date"]).dt.normalize()
panel["horizon"] = panel["horizon"].astype("int8")

panel = panel.loc[panel["horizon"].isin(HORIZONS_TO_RUN)].reset_index(drop=True)
panel = panel.loc[panel["Antall_total"].notna()].reset_index(drop=True)

# Forecast-origin-aware split; gap rows outside train/test definitions are dropped.
train_mask_full = panel["target_date"] <= TRAIN_END_DATE
test_mask_full = (panel["origin_date"] >= TEST_START_DATE) & (panel["target_date"] <= TEST_END_DATE)

panel = panel.loc[train_mask_full | test_mask_full].reset_index(drop=True)

print(f"Panel rows after horizon, target, and split filtering: {len(panel):,}")
print(
    f"Selected columns ({len(selected_columns)}): "
    f"{selected_columns[:8]}{'...' if len(selected_columns) > 8 else ''}"
)
print(f"Memory (approx): {panel.memory_usage(deep=True).sum() / 1_000_000_000:.3f} GB")


## Model factory and pipeline builder

Each model family is created with fixed starting hyperparameters. Quick mode uses smaller estimator counts than full mode. Pipelines combine one-hot encoding for categorical features, passthrough numeric features, and the selected regressor.


In [ ]:
def make_random_forest(run_mode):
    n_estimators = 100 if run_mode == "quick" else 200
    max_depth = 16 if run_mode == "quick" else None
    return RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )


def make_xgboost(run_mode):
    from xgboost import XGBRegressor
    n_estimators = 200 if run_mode == "quick" else 400
    return XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        n_estimators=n_estimators,
        learning_rate=0.1,
        max_depth=6,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbosity=0,
    )


def make_lightgbm(run_mode):
    from lightgbm import LGBMRegressor
    n_estimators = 200 if run_mode == "quick" else 400
    return LGBMRegressor(
        objective="regression",
        n_estimators=n_estimators,
        learning_rate=0.1,
        num_leaves=63,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=-1,
    )


def make_catboost(run_mode):
    from catboost import CatBoostRegressor
    iterations = 200 if run_mode == "quick" else 500
    return CatBoostRegressor(
        loss_function="RMSE",
        iterations=iterations,
        learning_rate=0.1,
        depth=6,
        random_seed=RANDOM_STATE,
        verbose=False,
        allow_writing_files=False,
    )


MODEL_FACTORY = {
    "random_forest": make_random_forest,
    "xgboost": make_xgboost,
    "lightgbm": make_lightgbm,
    "catboost": make_catboost,
}


def build_pipeline(model_family, numeric_columns, categorical_columns, run_mode):
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    transformer = ColumnTransformer(
        transformers=[
            ("num", "passthrough", numeric_columns),
            ("cat", encoder, categorical_columns),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )
    estimator = MODEL_FACTORY[model_family](run_mode)
    if model_family == "random_forest":
        # Random Forest uses dense matrices for this moderate one-hot design.
        from sklearn.preprocessing import FunctionTransformer
        densifier = FunctionTransformer(
            lambda x: x.toarray() if hasattr(x, "toarray") else np.asarray(x),
            accept_sparse=True,
        )
        return Pipeline([("preprocess", transformer), ("densify", densifier), ("model", estimator)])
    return Pipeline([("preprocess", transformer), ("model", estimator)])


## Metric functions

The notebook computes `RMSE`, `MAE`, `WAPE`, and bias directly. WAPE handles zero denominators explicitly, and plain MAPE is not used because the target can be zero.


In [ ]:
def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype="float64")
    y_pred = np.asarray(y_pred, dtype="float64")
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype="float64")
    y_pred = np.asarray(y_pred, dtype="float64")
    return float(np.mean(np.abs(y_true - y_pred)))


def wape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype="float64")
    y_pred = np.asarray(y_pred, dtype="float64")
    denom = float(np.sum(np.abs(y_true)))
    if denom <= 0:
        return float("nan")
    return float(np.sum(np.abs(y_true - y_pred)) / denom)


def bias(y_true, y_pred):
    y_true = np.asarray(y_true, dtype="float64")
    y_pred = np.asarray(y_pred, dtype="float64")
    return float(np.mean(y_pred - y_true))


def all_metrics(y_true, y_pred):
    return {
        "rmse": rmse(y_true, y_pred),
        "mae": mae(y_true, y_pred),
        "wape": wape(y_true, y_pred),
        "bias": bias(y_true, y_pred),
        "n": int(len(y_true)),
    }


## Benchmark loop

The benchmark iterates over horizons, feature sets, and available model families. For each combination it builds train/test matrices, fits the pipeline, predicts, clips negative predictions to zero, records pooled and category-level metrics, appends row-level predictions, and releases fitted objects after use.


In [ ]:
KEY_PRED_COLUMNS = ["AvdelingID", "Analyse_Kategori", "target_date", "origin_date", "horizon"]

metric_records = []
prediction_frames = []
model_summary_records = []

skipped_families = {fam: reason for fam, (avail, reason) in MODEL_AVAILABILITY.items() if not avail}

active_model_families = [fam for fam in MODEL_FAMILIES if MODEL_AVAILABILITY[fam][0]]
for fam in MODEL_FAMILIES:
    if not MODEL_AVAILABILITY[fam][0]:
        print(f"Skipping {fam}: {MODEL_AVAILABILITY[fam][1]}")

run_started_at = time.time()

for horizon in HORIZONS_TO_RUN:
    panel_h = panel.loc[panel["horizon"] == horizon].reset_index(drop=True)
    if len(panel_h) == 0:
        print(f"No rows for horizon {horizon}; skipping.")
        continue

    # Forecast-origin-aware chronological split.
    train_mask = panel_h["target_date"] <= TRAIN_END_DATE
    test_mask = (panel_h["origin_date"] >= TEST_START_DATE) & (
        panel_h["target_date"] <= TEST_END_DATE
    )
    train_panel = panel_h.loc[train_mask].reset_index(drop=True)
    test_panel = panel_h.loc[test_mask].reset_index(drop=True)

    train_rows = len(train_panel)
    test_rows = len(test_panel)
    print(f"\n--- Horizon {horizon}: train_rows={train_rows:,} test_rows={test_rows:,} ---")

    if train_rows == 0 or test_rows == 0:
        print("  Empty train or test for this horizon; skipping all models at this horizon.")
        continue

    # Raise if the forecast-origin holdout boundary is violated.
    assert train_panel["target_date"].max() <= TRAIN_END_DATE, (
        f"Horizon {horizon}: train target_date exceeds TRAIN_END_DATE"
    )
    assert test_panel["origin_date"].min() >= TEST_START_DATE, (
        f"Horizon {horizon}: test origin_date precedes TEST_START_DATE"
    )
    assert test_panel["target_date"].max() <= TEST_END_DATE, (
        f"Horizon {horizon}: test target_date exceeds TEST_END_DATE"
    )
    assert bool((test_panel["origin_date"] > TRAIN_END_DATE).all()), (
        f"Horizon {horizon}: at least one test row has origin_date <= TRAIN_END_DATE"
    )

    y_train = train_panel["Antall_total"].astype("float32").to_numpy()
    y_test = test_panel["Antall_total"].astype("float32").to_numpy()

    for feature_set_label in FEATURE_SETS_TO_RUN:
        numeric_cols = feature_set_numeric[feature_set_label]
        categorical_cols = feature_set_categorical[feature_set_label]
        feature_cols = numeric_cols + categorical_cols
        X_train = train_panel[feature_cols]
        X_test = test_panel[feature_cols]

        for model_family in MODEL_FAMILIES:
            available, reason = MODEL_AVAILABILITY[model_family]
            if not available:
                model_summary_records.append({
                    "run_mode": RUN_MODE,
                    "model_family": model_family,
                    "feature_set": feature_set_label,
                    "horizon": horizon,
                    "train_rows": train_rows,
                    "test_rows": test_rows,
                    "n_features": len(feature_cols),
                    "n_numeric_features": len(numeric_cols),
                    "n_categorical_features": len(categorical_cols),
                    "training_time_seconds": float("nan"),
                    "prediction_time_seconds": float("nan"),
                    "n_clipped_predictions": 0,
                    "share_clipped_predictions": float("nan"),
                    "status": "skipped_unavailable",
                    "skip_reason": reason,
                })
                continue

            print(f"  Fit: family={model_family} feature_set={feature_set_label}")
            try:
                pipeline = build_pipeline(model_family, numeric_cols, categorical_cols, RUN_MODE)
                t0 = time.time()
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    pipeline.fit(X_train, y_train)
                training_time = time.time() - t0

                t1 = time.time()
                y_pred = pipeline.predict(X_test)
                prediction_time = time.time() - t1

                y_pred = np.asarray(y_pred, dtype="float64")
                if CLIP_NEGATIVE_PREDICTIONS:
                    n_clipped = int((y_pred < 0).sum())
                    y_pred = np.clip(y_pred, 0.0, None)
                else:
                    n_clipped = 0
                share_clipped = n_clipped / max(len(y_pred), 1)

                metrics_pooled = all_metrics(y_test, y_pred)
                metric_records.append({
                    "run_mode": RUN_MODE,
                    "model_family": model_family,
                    "feature_set": feature_set_label,
                    "horizon": horizon,
                    "Analyse_Kategori": "__POOLED__",
                    **metrics_pooled,
                })

                cats = test_panel["Analyse_Kategori"].astype(str).to_numpy()
                for category in pd.Series(cats).unique():
                    mask = cats == category
                    if mask.sum() == 0:
                        continue
                    metrics_cat = all_metrics(y_test[mask], y_pred[mask])
                    metric_records.append({
                        "run_mode": RUN_MODE,
                        "model_family": model_family,
                        "feature_set": feature_set_label,
                        "horizon": horizon,
                        "Analyse_Kategori": category,
                        **metrics_cat,
                    })

                pred_frame = test_panel[KEY_PRED_COLUMNS].copy()
                pred_frame["run_mode"] = RUN_MODE
                pred_frame["model_family"] = model_family
                pred_frame["feature_set"] = feature_set_label
                pred_frame["y_true"] = y_test
                pred_frame["y_pred"] = y_pred.astype("float32")
                prediction_frames.append(pred_frame)

                model_summary_records.append({
                    "run_mode": RUN_MODE,
                    "model_family": model_family,
                    "feature_set": feature_set_label,
                    "horizon": horizon,
                    "train_rows": train_rows,
                    "test_rows": test_rows,
                    "n_features": len(feature_cols),
                    "n_numeric_features": len(numeric_cols),
                    "n_categorical_features": len(categorical_cols),
                    "training_time_seconds": float(training_time),
                    "prediction_time_seconds": float(prediction_time),
                    "n_clipped_predictions": int(n_clipped),
                    "share_clipped_predictions": float(share_clipped),
                    "status": "ok",
                    "skip_reason": "",
                })

                del pipeline, y_pred, pred_frame
                gc.collect()
            except Exception as exc:  # pragma: no cover
                model_summary_records.append({
                    "run_mode": RUN_MODE,
                    "model_family": model_family,
                    "feature_set": feature_set_label,
                    "horizon": horizon,
                    "train_rows": train_rows,
                    "test_rows": test_rows,
                    "n_features": len(feature_cols),
                    "n_numeric_features": len(numeric_cols),
                    "n_categorical_features": len(categorical_cols),
                    "training_time_seconds": float("nan"),
                    "prediction_time_seconds": float("nan"),
                    "n_clipped_predictions": 0,
                    "share_clipped_predictions": float("nan"),
                    "status": "failed",
                    "skip_reason": f"{type(exc).__name__}: {exc}",
                })
                print(f"  FAILED: {model_family} {feature_set_label} h={horizon}: {exc}")

    del panel_h, train_panel, test_panel, y_train, y_test
    gc.collect()

run_elapsed = time.time() - run_started_at
print(f"\nBenchmark loop finished in {run_elapsed/60:.2f} min over {len(HORIZONS_TO_RUN)} horizons.")


## Metrics, gain summary, and feature-set summary

This step aggregates recorded metrics, summarizes feature-set composition, and computes weather-gain comparisons against M1 within the same model family, horizon, and category. It also attaches horizon-tier labels so later reporting can separate archived MEPS evidence from synthetic medium-range scenario evidence.


In [ ]:
metrics_df = pd.DataFrame(metric_records)
model_summary_df = pd.DataFrame(model_summary_records)


def _frac_assert_finite(df, columns):
    for col in columns:
        bad = (~np.isfinite(df[col])).sum()
        if bad:
            print(f"WARNING: column {col!r} has {bad} non-finite metric values")


if not metrics_df.empty:
    _frac_assert_finite(metrics_df, ["rmse", "mae"])

feature_set_summary_rows = []
for label in FEATURE_SETS_TO_RUN:
    columns = feature_sets[label]
    # Current weather feature groups plus legacy aliases for older registry exports.
    weather_groups = {
        "operational_weather_point",
        "operational_weather_uncertainty_probability_interval",
        "historical_weather",
        "realised_weather_perfect_information",
        # Older registry aliases.
        "synthetic_weather_point",
        "synthetic_weather_uncertainty",
        "synthetic_weather_probability",
    }
    weather_columns = [
        c
        for c in columns
        if c in set(feature_registry.loc[feature_registry["feature_group"].isin(weather_groups), "column"])
    ]
    leakage_columns_in_set = [
        c for c in columns if c in set(feature_registry.loc[feature_registry["leakage_risk"], "column"])
    ]
    feature_set_summary_rows.append(
        {
            "run_mode": RUN_MODE,
            "feature_set": label,
            "n_features": len(columns),
            "n_numeric_features": len(feature_set_numeric[label]),
            "n_categorical_features": len(feature_set_categorical[label]),
            "weather_feature_count": len(weather_columns),
            "leakage_risk_count": len(leakage_columns_in_set),
            "feature_columns": ";".join(columns),
        }
    )

feature_set_summary_df = pd.DataFrame(feature_set_summary_rows)


# Gain summary against M1 within model_family, horizon, and Analyse_Kategori.
def _build_gain_summary(metrics_df_):
    if metrics_df_.empty:
        return pd.DataFrame()
    pivot_cols = ["model_family", "horizon", "Analyse_Kategori"]
    base = metrics_df_.loc[metrics_df_["feature_set"] == "M1", pivot_cols + ["rmse", "mae", "wape"]].rename(
        columns={"rmse": "rmse_M1", "mae": "mae_M1", "wape": "wape_M1"}
    )
    other = metrics_df_.loc[
        metrics_df_["feature_set"].isin(["M2", "M3", "M4"]),
        pivot_cols + ["feature_set", "rmse", "mae", "wape"],
    ]
    merged = other.merge(base, on=pivot_cols, how="left")
    merged["delta_rmse"] = merged["rmse"] - merged["rmse_M1"]
    merged["delta_mae"] = merged["mae"] - merged["mae_M1"]
    merged["delta_wape"] = merged["wape"] - merged["wape_M1"]
    merged["pct_improvement_rmse"] = -merged["delta_rmse"] / merged["rmse_M1"]
    merged["pct_improvement_mae"] = -merged["delta_mae"] / merged["mae_M1"]
    merged["pct_improvement_wape"] = -merged["delta_wape"] / merged["wape_M1"]
    merged.insert(0, "run_mode", RUN_MODE)
    return merged


gain_summary_df = _build_gain_summary(metrics_df)

# Attach horizon-tier labels for MEPS evidence (h=0-h=2)
# versus synthetic scenario evidence (h=3-h=10).
if not metrics_df.empty:
    metrics_df["horizon_tier"] = metrics_df["horizon"].map(horizon_tier_for)
if not gain_summary_df.empty:
    gain_summary_df["horizon_tier"] = gain_summary_df["horizon"].map(horizon_tier_for)


# Weather-value decomposition.
def _build_weather_value_decomposition(metrics_df_):
    """Compute pairwise feature-set metric deltas for thesis interpretation.

    Pairs (reference -> comparison):
        M1 -> M2   point-forecast weather value
        M2 -> M4   incremental value of uncertainty/probability features
        M2 -> M3   gap from point forecast to perfect information
        M4 -> M3   gap from uncertainty-aware forecast features to perfect information

    Sign convention:
        metric_delta       = comparison_value - reference_value
        metric_improvement = reference_value - comparison_value (positive = comparison wins)
        pct_improvement    = metric_improvement / reference_value
    """
    if metrics_df_.empty:
        return pd.DataFrame()
    PAIRS = [
        ("M1", "M2", "point_forecast_weather_value"),
        ("M2", "M4", "uncertainty_features_value"),
        ("M2", "M3", "gap_from_point_forecast_to_perfect_information"),
        ("M4", "M3", "gap_from_uncertainty_aware_to_perfect_information"),
    ]
    METRIC_COLS = ["rmse", "mae", "wape"]
    pivot_cols = ["model_family", "horizon", "Analyse_Kategori"]
    rows = []
    for ref_set, comp_set, _label in PAIRS:
        ref = metrics_df_.loc[metrics_df_["feature_set"] == ref_set, pivot_cols + METRIC_COLS].rename(
            columns={m: f"ref_{m}" for m in METRIC_COLS}
        )
        comp = metrics_df_.loc[metrics_df_["feature_set"] == comp_set, pivot_cols + METRIC_COLS].rename(
            columns={m: f"comp_{m}" for m in METRIC_COLS}
        )
        merged = ref.merge(comp, on=pivot_cols, how="inner")
        for metric in METRIC_COLS:
            ref_val = merged[f"ref_{metric}"].astype("float64")
            comp_val = merged[f"comp_{metric}"].astype("float64")
            delta = comp_val - ref_val
            improvement = ref_val - comp_val
            with np.errstate(divide="ignore", invalid="ignore"):
                pct = np.where(ref_val.abs() > 0, improvement / ref_val, np.nan)
            block = merged[pivot_cols].copy()
            block["run_mode"] = RUN_MODE
            block["reference_feature_set"] = ref_set
            block["comparison_feature_set"] = comp_set
            block["metric"] = metric
            block["reference_value"] = ref_val
            block["comparison_value"] = comp_val
            block["metric_delta"] = delta
            block["metric_improvement"] = improvement
            block["pct_improvement"] = pct
            rows.append(block)
    if not rows:
        return pd.DataFrame()
    out = pd.concat(rows, ignore_index=True)
    out["horizon_tier"] = out["horizon"].map(horizon_tier_for)
    column_order = [
        "run_mode",
        "model_family",
        "horizon",
        "horizon_tier",
        "Analyse_Kategori",
        "reference_feature_set",
        "comparison_feature_set",
        "metric",
        "reference_value",
        "comparison_value",
        "metric_delta",
        "metric_improvement",
        "pct_improvement",
    ]
    return out[column_order]


weather_value_decomposition_df = _build_weather_value_decomposition(metrics_df)


# M4-vs-M2 diagnostic.
def _support_label(rmse_imp, mae_imp, wape_imp):
    """Classify M4 vs M2 support based on RMSE/MAE/WAPE improvements."""
    vals = [rmse_imp, mae_imp, wape_imp]
    finite = [v for v in vals if pd.notna(v)]
    if not finite:
        return "missing"
    positives = sum(1 for v in finite if v > 1e-9)
    negatives = sum(1 for v in finite if v < -1e-9)
    if positives == len(finite):
        return "strong_support"
    if negatives == len(finite):
        return "no_support"
    if positives >= 2:
        return "moderate_support"
    if positives == 1 and negatives == 0:
        return "weak_support"
    if positives >= 1 and negatives >= 1:
        return "mixed"
    return "weak_support"


def _build_m4_vs_m2_diagnostic(metrics_df_):
    if metrics_df_.empty:
        return pd.DataFrame()
    pivot_cols = ["model_family", "horizon", "Analyse_Kategori"]
    METRIC_COLS = ["rmse", "mae", "wape"]
    m2 = metrics_df_.loc[metrics_df_["feature_set"] == "M2", pivot_cols + METRIC_COLS].rename(
        columns={m: f"m2_{m}" for m in METRIC_COLS}
    )
    m4 = metrics_df_.loc[metrics_df_["feature_set"] == "M4", pivot_cols + METRIC_COLS].rename(
        columns={m: f"m4_{m}" for m in METRIC_COLS}
    )
    if m2.empty or m4.empty:
        return pd.DataFrame()
    merged = m2.merge(m4, on=pivot_cols, how="inner")
    for metric in METRIC_COLS:
        merged[f"m4_improvement_{metric}"] = (
            merged[f"m2_{metric}"].astype("float64") - merged[f"m4_{metric}"].astype("float64")
        )
    merged["horizon_tier"] = merged["horizon"].map(horizon_tier_for)
    merged["run_mode"] = RUN_MODE
    merged["support_label"] = merged.apply(
        lambda r: _support_label(
            r["m4_improvement_rmse"],
            r["m4_improvement_mae"],
            r["m4_improvement_wape"],
        ),
        axis=1,
    )
    column_order = [
        "run_mode",
        "model_family",
        "horizon",
        "horizon_tier",
        "Analyse_Kategori",
        "m2_rmse",
        "m4_rmse",
        "m4_improvement_rmse",
        "m2_mae",
        "m4_mae",
        "m4_improvement_mae",
        "m2_wape",
        "m4_wape",
        "m4_improvement_wape",
        "support_label",
    ]
    return merged[column_order]


m4_vs_m2_diagnostic_df = _build_m4_vs_m2_diagnostic(metrics_df)


def _build_m4_overfit_risk_summary(diagnostic_df):
    """Summarise share of horizons where M4 improves over M2, by tier and category."""
    if diagnostic_df.empty:
        return pd.DataFrame()
    rows = []
    pooled_marker = "__POOLED__"
    is_pooled = diagnostic_df["Analyse_Kategori"] == pooled_marker
    pooled = diagnostic_df.loc[is_pooled].copy()

    def _share(frame):
        if frame.empty:
            return float("nan")
        improving = (frame["m4_improvement_rmse"] > 0).astype("float64")
        return float(improving.mean())

    families = sorted(diagnostic_df["model_family"].unique())
    for fam in families:
        sub = pooled.loc[pooled["model_family"] == fam]
        rows.append(
            {
                "run_mode": RUN_MODE,
                "model_family": fam,
                "scope": "all_horizons_pooled_categories",
                "horizon_tier": "all",
                "Analyse_Kategori": pooled_marker,
                "n_horizons": int(len(sub)),
                "share_m4_improves_rmse": _share(sub),
            }
        )
        for tier in ("actual_meps_h0_h2", "synthetic_scenario_h3_h10"):
            sub_t = sub.loc[sub["horizon_tier"] == tier]
            rows.append(
                {
                    "run_mode": RUN_MODE,
                    "model_family": fam,
                    "scope": "by_horizon_tier_pooled_categories",
                    "horizon_tier": tier,
                    "Analyse_Kategori": pooled_marker,
                    "n_horizons": int(len(sub_t)),
                    "share_m4_improves_rmse": _share(sub_t),
                }
            )
    # Use non-pooled rows for category diagnostics.
    by_cat = diagnostic_df.loc[~is_pooled].copy()
    for fam in families:
        for category in sorted(by_cat["Analyse_Kategori"].dropna().unique()):
            sub = by_cat.loc[
                (by_cat["model_family"] == fam) & (by_cat["Analyse_Kategori"] == category)
            ]
            rows.append(
                {
                    "run_mode": RUN_MODE,
                    "model_family": fam,
                    "scope": "by_category_all_horizons",
                    "horizon_tier": "all",
                    "Analyse_Kategori": str(category),
                    "n_horizons": int(len(sub)),
                    "share_m4_improves_rmse": _share(sub),
                }
            )
    return pd.DataFrame(rows)


m4_overfit_risk_df = _build_m4_overfit_risk_summary(m4_vs_m2_diagnostic_df)


print(f"metrics rows: {len(metrics_df):,}")
print(f"model summary rows: {len(model_summary_df):,}")
print(f"feature set summary rows: {len(feature_set_summary_df):,}")
print(f"gain summary rows: {len(gain_summary_df):,}")
print(f"weather-value decomposition rows: {len(weather_value_decomposition_df):,}")
print(f"M4 vs M2 diagnostic rows: {len(m4_vs_m2_diagnostic_df):,}")
print(f"M4 overfitting-risk summary rows: {len(m4_overfit_risk_df):,}")


## Save outputs

The notebook writes metrics, row-level predictions, model summary, feature-set summary, and gain summary. Predictions are stored as a single concatenated parquet file.


In [ ]:
if prediction_frames:
    predictions_df = pd.concat(prediction_frames, ignore_index=True)
else:
    predictions_df = pd.DataFrame(
        columns=[
            "run_mode", "model_family", "feature_set",
            *KEY_PRED_COLUMNS,
            "y_true", "y_pred",
        ]
    )

# Retain horizon_tier for downstream thesis filtering.
if not predictions_df.empty and "horizon" in predictions_df.columns:
    predictions_df["horizon_tier"] = predictions_df["horizon"].map(horizon_tier_for)

metrics_df.to_csv(METRICS_PATH, index=False)
predictions_df.to_parquet(PREDICTIONS_PATH, index=False)
model_summary_df.to_csv(MODEL_SUMMARY_PATH, index=False)
feature_set_summary_df.to_csv(FEATURE_SET_SUMMARY_PATH, index=False)
gain_summary_df.to_csv(GAIN_SUMMARY_PATH, index=False)
weather_value_decomposition_df.to_csv(WEATHER_VALUE_DECOMPOSITION_PATH, index=False)
m4_vs_m2_diagnostic_df.to_csv(M4_VS_M2_DIAGNOSTIC_PATH, index=False)
m4_overfit_risk_df.to_csv(M4_OVERFIT_RISK_PATH, index=False)

print(f"Saved: {project_relative(METRICS_PATH)}")
print(f"Saved: {project_relative(PREDICTIONS_PATH)} (rows={len(predictions_df):,})")
print(f"Saved: {project_relative(MODEL_SUMMARY_PATH)}")
print(f"Saved: {project_relative(FEATURE_SET_SUMMARY_PATH)}")
print(f"Saved: {project_relative(GAIN_SUMMARY_PATH)}")
print(f"Saved: {project_relative(WEATHER_VALUE_DECOMPOSITION_PATH)}")
print(f"Saved: {project_relative(M4_VS_M2_DIAGNOSTIC_PATH)}")
print(f"Saved: {project_relative(M4_OVERFIT_RISK_PATH)}")


## Model-family selection tables

This step builds thesis-ready model-family comparison tables from the same metrics and model-summary objects created above. The output is neutral: it summarizes predictive performance and weather-value decomposition for tree-based families, but does not select a final model. M3 is excluded from the best operational feature-set comparison because realised target-day weather is not available at forecast origin. Quick-mode runs write `_quick`-suffixed files and do not overwrite canonical full-mode files.


In [ ]:
# Model-family selection tables: neutral performance summaries with no recommendation field.
# M3 is excluded from operational feature-set selection because it uses realised target-day weather.
# Quick-mode outputs use *_quick suffixes and do not overwrite canonical full-mode files.

DOCS_DIR = PROJECT_ROOT / "docs"
DOCS_DIR.mkdir(parents=True, exist_ok=True)

if RUN_MODE == "full":
    MFS_CSV_PATH = OUTPUT_ML_MODELS_DIR / "model_family_selection_summary.csv"
    MFS_HTML_PATH = OUTPUT_ML_MODELS_DIR / "model_family_selection_summary.html"
    MFS_MD_PATH = DOCS_DIR / "THESIS_MODEL_SELECTION_TABLE.md"
else:
    MFS_CSV_PATH = OUTPUT_ML_MODELS_DIR / "model_family_selection_summary_quick.csv"
    MFS_HTML_PATH = OUTPUT_ML_MODELS_DIR / "model_family_selection_summary_quick.html"
    MFS_MD_PATH = DOCS_DIR / "THESIS_MODEL_SELECTION_TABLE_quick.md"

# Factual role labels, not recommendations.
_MFS_NEUTRAL_ROLE = {
    "random_forest": "non-boosting tree baseline",
    "xgboost":       "gradient boosting benchmark",
    "lightgbm":      "histogram-based gradient boosting benchmark",
    "catboost":      "categorical boosting benchmark",
}

# Display names for rendered model-family tables.
_MFS_DISPLAY_NAME = {
    "random_forest": "Random Forest",
    "xgboost":       "XGBoost",
    "lightgbm":      "LightGBM",
    "catboost":      "CatBoost",
}


def _mfs_safe_mean(values):
    arr = np.asarray(values, dtype="float64")
    arr = arr[np.isfinite(arr)]
    return float(arr.mean()) if arr.size else float("nan")


def _mfs_build_long_table(metrics_df_, model_summary_df_):
    if metrics_df_.empty:
        return pd.DataFrame()
    pooled = metrics_df_.loc[metrics_df_["Analyse_Kategori"] == "__POOLED__"].copy()
    if pooled.empty:
        return pd.DataFrame()
    families = sorted(pooled["model_family"].unique())
    horizons = sorted(pooled["horizon"].unique())
    rows = []
    for fam in families:
        sub = pooled.loc[pooled["model_family"] == fam]
        per_fs_rmse = {fs: _mfs_safe_mean(sub.loc[sub["feature_set"] == fs, "rmse"].values)
                       for fs in ["M1", "M2", "M3", "M4"]}
        per_fs_mae  = {fs: _mfs_safe_mean(sub.loc[sub["feature_set"] == fs, "mae"].values)
                       for fs in ["M1", "M2", "M3", "M4"]}
        per_fs_wape = {fs: _mfs_safe_mean(sub.loc[sub["feature_set"] == fs, "wape"].values)
                       for fs in ["M1", "M2", "M3", "M4"]}

        # Operational feature-set comparison excludes non-operational M3.
        operational_rmse = {fs: per_fs_rmse[fs] for fs in ["M1", "M2", "M4"]
                            if np.isfinite(per_fs_rmse[fs])}
        if operational_rmse:
            best_op = min(operational_rmse, key=operational_rmse.get)
        else:
            best_op = ""

        # Improvements versus M1, equal-weighted across horizons and tiers.
        def _imp_pct(fs_target, allowed_tier=None):
            deltas = []
            for h in horizons:
                if allowed_tier is not None and horizon_tier_for(h) != allowed_tier:
                    continue
                r1 = sub.loc[(sub["feature_set"] == "M1") & (sub["horizon"] == h), "rmse"]
                rx = sub.loc[(sub["feature_set"] == fs_target) & (sub["horizon"] == h), "rmse"]
                if len(r1) == 0 or len(rx) == 0:
                    continue
                deltas.append((float(r1.iloc[0]) - float(rx.iloc[0])) / float(r1.iloc[0]))
            return (float(np.mean(deltas)) if deltas else float("nan")) * 100.0

        def _m4_vs_m2_pct():
            deltas = []
            for h in horizons:
                r2 = sub.loc[(sub["feature_set"] == "M2") & (sub["horizon"] == h), "rmse"]
                r4 = sub.loc[(sub["feature_set"] == "M4") & (sub["horizon"] == h), "rmse"]
                if len(r2) == 0 or len(r4) == 0:
                    continue
                deltas.append((float(r2.iloc[0]) - float(r4.iloc[0])) / float(r2.iloc[0]))
            return (float(np.mean(deltas)) if deltas else float("nan")) * 100.0

        ms_sub = model_summary_df_.loc[model_summary_df_["model_family"] == fam]
        ok_count = int((ms_sub["status"] == "ok").sum())
        failed_count = int((ms_sub["status"] != "ok").sum())
        if failed_count == 0 and ok_count > 0:
            status_str = "completed"
        elif ok_count == 0:
            status_str = "failed"
        else:
            status_str = f"partial ({failed_count} failed)"

        ok_rows = ms_sub.loc[ms_sub["status"] == "ok"]
        if not ok_rows.empty and "training_time_seconds" in ok_rows.columns:
            train_minutes = float(ok_rows["training_time_seconds"].sum() / 60.0)
        else:
            train_minutes = float("nan")

        # Notes remain factual and non-recommendatory.
        note_pieces = []
        if best_op and np.isfinite(operational_rmse[best_op]):
            note_pieces.append(
                f"Best operational feature set is {best_op} at mean pooled RMSE {operational_rmse[best_op]:.3f}"
            )
        rows.append({
            "model_family": fam,
            "benchmark_status": status_str,
            "completed_configurations": ok_count,
            "failed_configurations": failed_count,
            "mean_rmse_M1": per_fs_rmse["M1"],
            "mean_rmse_M2": per_fs_rmse["M2"],
            "mean_rmse_M3": per_fs_rmse["M3"],
            "mean_rmse_M4": per_fs_rmse["M4"],
            "best_operational_feature_set_by_rmse": best_op,
            "mean_mae_best_operational":  per_fs_mae.get(best_op, float("nan")),
            "mean_wape_best_operational": per_fs_wape.get(best_op, float("nan")),
            "M2_vs_M1_rmse_improvement_pct": _imp_pct("M2"),
            "M4_vs_M2_rmse_improvement_pct": _m4_vs_m2_pct(),
            "actual_meps_h0_h2_M2_vs_M1_rmse_improvement_pct":
                _imp_pct("M2", allowed_tier="actual_meps_h0_h2"),
            "synthetic_scenario_h3_h10_M2_vs_M1_rmse_improvement_pct":
                _imp_pct("M2", allowed_tier="synthetic_scenario_h3_h10"),
            "training_time_total_minutes": train_minutes,
            "methodological_role": _MFS_NEUTRAL_ROLE.get(fam, "tree-based benchmark"),
            "notes": "; ".join(note_pieces),
        })
    out = pd.DataFrame(rows).sort_values("mean_rmse_M2", na_position="last").reset_index(drop=True)
    return out


def _mfs_format_number(value, kind="rmse"):
    if value is None:
        return ""
    try:
        v = float(value)
    except (TypeError, ValueError):
        return str(value)
    if not np.isfinite(v):
        return ""
    if kind == "rmse" or kind == "mae":
        return f"{v:.3f}"
    if kind == "wape":
        return f"{v:.4f}"
    if kind == "pct":
        return f"{v:.2f}%"
    if kind == "minutes":
        return f"{v:.1f}"
    if kind == "int":
        return f"{int(round(v)):,}"
    return f"{v:.3f}"


def _mfs_format_for_table(df, formats):
    out = df.copy()
    for col, kind in formats.items():
        if col in out.columns:
            out[col] = out[col].map(lambda x, _k=kind: _mfs_format_number(x, _k))
    return out.where(pd.notna(out), "")


def _mfs_render_predictive_table(long_df):
    cols = [
        ("model_family",                            "Model family"),
        ("benchmark_status",                        "Benchmark status"),
        ("completed_configurations",                "Completed"),
        ("failed_configurations",                   "Failed"),
        ("mean_rmse_M1",                            "Mean RMSE M1"),
        ("mean_rmse_M2",                            "Mean RMSE M2"),
        ("mean_rmse_M3",                            "Mean RMSE M3"),
        ("mean_rmse_M4",                            "Mean RMSE M4"),
        ("best_operational_feature_set_by_rmse",    "Best operational feature set (RMSE)"),
        ("mean_mae_best_operational",               "Mean MAE best operational"),
        ("mean_wape_best_operational",              "Mean WAPE best operational"),
    ]
    formats = {
        "completed_configurations":    "int",
        "failed_configurations":       "int",
        "mean_rmse_M1":                "rmse",
        "mean_rmse_M2":                "rmse",
        "mean_rmse_M3":                "rmse",
        "mean_rmse_M4":                "rmse",
        "mean_mae_best_operational":   "mae",
        "mean_wape_best_operational":  "wape",
    }
    pretty = long_df[[c for c, _ in cols]].copy()
    pretty["model_family"] = pretty["model_family"].map(lambda f: _MFS_DISPLAY_NAME.get(f, f))
    pretty = _mfs_format_for_table(pretty, formats)
    pretty.columns = [label for _, label in cols]
    return pretty


def _mfs_render_practical_table(long_df):
    cols = [
        ("model_family",                                                  "Model family"),
        ("M2_vs_M1_rmse_improvement_pct",                                 "M2 vs M1 RMSE %"),
        ("M4_vs_M2_rmse_improvement_pct",                                 "M4 vs M2 RMSE %"),
        ("actual_meps_h0_h2_M2_vs_M1_rmse_improvement_pct",               "M2 vs M1 RMSE % (h=0-h=2)"),
        ("synthetic_scenario_h3_h10_M2_vs_M1_rmse_improvement_pct",       "M2 vs M1 RMSE % (h=3-h=10)"),
        ("training_time_total_minutes",                                   "Total train time (min)"),
        ("methodological_role",                                           "Methodological role"),
        ("notes",                                                         "Notes"),
    ]
    formats = {
        "M2_vs_M1_rmse_improvement_pct":                            "pct",
        "M4_vs_M2_rmse_improvement_pct":                            "pct",
        "actual_meps_h0_h2_M2_vs_M1_rmse_improvement_pct":          "pct",
        "synthetic_scenario_h3_h10_M2_vs_M1_rmse_improvement_pct":  "pct",
        "training_time_total_minutes":                              "minutes",
    }
    pretty = long_df[[c for c, _ in cols]].copy()
    pretty["model_family"] = pretty["model_family"].map(lambda f: _MFS_DISPLAY_NAME.get(f, f))
    pretty = _mfs_format_for_table(pretty, formats)
    pretty.columns = [label for _, label in cols]
    return pretty


def _mfs_table_css():
    return """
<style>
  @page { size: A4 landscape; margin: 12mm; }
  body { font-family: \"Times New Roman\", Times, serif; font-size: 9.5pt; margin: 0; padding: 16px; }
  .title    { text-align: center; font-size: 12pt; font-weight: bold; margin: 0 0 6px 0; }
  .subtitle { text-align: left;   font-size: 10pt; font-weight: bold; margin: 14px 0 4px 0; }
  .note     { margin-top: 6px; font-size: 9.5pt; text-align: left; }
  table {
    border-collapse: collapse;
    width: 100%;
    table-layout: auto;
    border-top: 1.2px solid #000;
    border-bottom: 1.2px solid #000;
    margin: 0 0 10px 0;
  }
  th, td { border: none; padding: 3px 6px; vertical-align: top; }
  thead th { text-align: center; font-weight: bold; padding-bottom: 4px; line-height: 1.05; }
  thead tr:last-child th { border-bottom: 1px solid #000; }
  tbody td { text-align: center; white-space: nowrap; }
  tbody td:first-child, tbody th:first-child { text-align: left; }
  tbody tr td:last-child { white-space: normal; text-align: left; }
  @media print {
    thead { display: table-row-group; }
    tfoot { display: table-row-group; }
  }
</style>
"""


def _mfs_render_html(predictive_pretty, practical_pretty, run_mode):
    css = _mfs_table_css()
    pred_html = predictive_pretty.to_html(index=False, border=0, escape=True, na_rep="")
    prac_html = practical_pretty.to_html(index=False, border=0, escape=True, na_rep="")
    title = f"Model-family selection summary ({run_mode} mode)"
    notes = (
        "M3 is non-operational and is not eligible for the best operational feature set column "
        "because realised target-day weather is not known at the forecast origin. "
        "Horizons h=0-h=2 use actual deterministic MEPS forecasts; h=3-h=10 use the climatology-drift "
        "synthetic-scenario emulator. Positive improvement percentages indicate lower error relative to the "
        "reference model. The table is descriptive; it does not recommend a model family."
    )
    return (
        "<html><head>"
        + css
        + "</head><body>"
        + f'<div class=\"title\">{_mfs_html_escape(title)}</div>'
        + '<div class=\"subtitle\">Table 1 - Predictive performance by model family</div>'
        + pred_html
        + '<div class=\"subtitle\">Table 2 - Weather-value and practical comparison</div>'
        + prac_html
        + f'<div class=\"note\">{_mfs_html_escape(notes)}</div>'
        + "</body></html>"
    )


def _mfs_render_markdown_table(pretty_df, title):
    headers = list(pretty_df.columns)
    out_lines = []
    out_lines.append(f"### {title}")
    out_lines.append("")
    out_lines.append("| " + " | ".join(headers) + " |")
    out_lines.append("| " + " | ".join("---" for _ in headers) + " |")
    for _, row in pretty_df.iterrows():
        out_lines.append("| " + " | ".join(str(v) for v in row.tolist()) + " |")
    return "\n".join(out_lines)


def _mfs_render_markdown(long_df, predictive_pretty, practical_pretty, run_mode):
    timestamp = pd.Timestamp.now().isoformat(timespec="seconds")
    pred_md = _mfs_render_markdown_table(predictive_pretty, "Table 1 - Predictive performance by model family")
    prac_md = _mfs_render_markdown_table(practical_pretty, "Table 2 - Weather-value and practical comparison")
    families_seen = ", ".join(
        _MFS_DISPLAY_NAME.get(f, f) for f in long_df["model_family"].tolist()
    ) if not long_df.empty else "(none)"
    body = [
        "# Thesis Model-Family Selection Table",
        "",
        "Generated automatically by notebook 07 (`Model-Family Selection Tables` cell) when run in "
        f"`RUN_MODE = \"{run_mode}\"`. The numbers below come directly from the benchmark outputs "
        "produced in the same notebook run: `benchmark_metrics_*.csv` and `benchmark_model_summary_*.csv` "
        "in `outputs/ml_models/`. Holdout: forecast-origin-aware chronological split.",
        "",
        f"Generated: {timestamp}",
        f"Model families present in this run: {families_seen}",
        "",
        "## Tables",
        "",
        pred_md,
        "",
        prac_md,
        "",
        "### Notes",
        "",
        "- M3 is a non-operational realised-weather point-information benchmark; it is excluded from the "
        "  best operational feature set column because realised target-day weather is not available at the "
        "  forecast origin in production.",
        "- Horizons h=0-h=2 use actual deterministic MEPS forecasts. Horizons h=3-h=10 use the "
        "  climatology-drift synthetic-scenario emulator from notebook 04.",
        "- Positive improvement percentages indicate lower error relative to the reference model "
        "  (M1 for M2 / M4, and M2 for M4-vs-M2).",
        "- Mean RMSE and MAE are equal-weighted means across the eleven horizons (h=0..h=10) on the "
        "  pooled-categories view. WAPE uses the same averaging convention.",
        "- Training time is the sum of `training_time_seconds` across all `status == \"ok\"` rows for "
        "  the family; it is not directly comparable across families if hyperparameters differ.",
        "",
        "## Author decision text to complete",
        "",
        "The table is descriptive. Authors complete the paragraph below in the thesis:",
        "",
        "> Based on the full benchmark results, we choose **[MODEL FAMILY]** as the primary model family "
        "for the main analysis. The choice is motivated by **[predictive performance / robustness / "
        "computational efficiency / interpretability]**. **[OTHER MODEL FAMILY]** is retained as a "
        "robustness comparison, while **[OTHER MODEL FAMILY]** serves as **[role]**. M3 is treated only "
        "as a non-operational realised-weather benchmark and is not eligible as an operational model.",
        "",
        "The notebook deliberately does not pre-fill this paragraph; the thesis authors choose the model "
        "family.",
        "",
        "## Underlying outputs",
        "",
        f"- Wide CSV: `{project_relative(MFS_CSV_PATH)}`",
        f"- Styled HTML: `{project_relative(MFS_HTML_PATH)}`",
        f"- Pooled and per-category metrics: `outputs/ml_models/benchmark_metrics_{run_mode}.csv`",
        f"- Pct-improvement-vs-M1 view: `outputs/ml_models/benchmark_gain_summary_{run_mode}.csv`",
        f"- Weather-value decomposition: `outputs/ml_models/benchmark_weather_value_decomposition_{run_mode}.csv`",
        f"- M4-vs-M2 diagnostic: `outputs/ml_models/benchmark_m4_vs_m2_diagnostic_{run_mode}.csv`",
        f"- M4 overfitting-risk summary: `outputs/ml_models/benchmark_m4_overfitting_risk_summary_{run_mode}.csv`",
        f"- Model-run audit: `outputs/ml_models/benchmark_model_summary_{run_mode}.csv`",
        f"- Feature-set audit: `outputs/ml_models/feature_set_summary_{run_mode}.csv`",
        "",
    ]
    return "\n".join(body)


model_family_selection_summary_df = _mfs_build_long_table(metrics_df, model_summary_df)
model_family_selection_summary_df.to_csv(MFS_CSV_PATH, index=False)
print(f"Saved: {project_relative(MFS_CSV_PATH)}")

if not model_family_selection_summary_df.empty:
    _mfs_predictive_pretty = _mfs_render_predictive_table(model_family_selection_summary_df)
    _mfs_practical_pretty = _mfs_render_practical_table(model_family_selection_summary_df)
    MFS_HTML_PATH.write_text(
        _mfs_render_html(_mfs_predictive_pretty, _mfs_practical_pretty, RUN_MODE),
        encoding="utf-8",
    )
    print(f"Saved: {project_relative(MFS_HTML_PATH)}")
    MFS_MD_PATH.write_text(
        _mfs_render_markdown(
            model_family_selection_summary_df,
            _mfs_predictive_pretty,
            _mfs_practical_pretty,
            RUN_MODE,
        ),
        encoding="utf-8",
    )
    print(f"Saved: {project_relative(MFS_MD_PATH)}")
else:
    print(
        "Skipped HTML/Markdown selection-table output: metrics_df produced no pooled rows "
        "(likely an empty benchmark run)."
    )


## Diagnostic displays

The final displays provide compact pivots of pooled metrics by model family, feature set, and horizon, plus training-time and skipped-run summaries.


In [ ]:
pooled = metrics_df.loc[metrics_df["Analyse_Kategori"] == "__POOLED__"]
if not pooled.empty:
    print("Pooled RMSE by model_family, feature_set, horizon:")
    display(
        pooled.pivot_table(
            index=["model_family", "feature_set"],
            columns="horizon",
            values="rmse",
            aggfunc="mean",
        ).round(3)
    )
    print("Pooled WAPE by model_family, feature_set, horizon:")
    display(
        pooled.pivot_table(
            index=["model_family", "feature_set"],
            columns="horizon",
            values="wape",
            aggfunc="mean",
        ).round(4)
    )

print()
print("Model summary status counts:")
display(model_summary_df["status"].value_counts().rename_axis("status").reset_index(name="count"))

failed_or_skipped = model_summary_df.loc[model_summary_df["status"] != "ok"]
if not failed_or_skipped.empty:
    print("Failed or skipped runs:")
    display(failed_or_skipped)

if not gain_summary_df.empty:
    print("Mean pct improvement vs M1 by feature_set and model_family (pooled across categories and horizons):")
    pooled_gain = gain_summary_df.loc[gain_summary_df["Analyse_Kategori"] == "__POOLED__"]
    display(
        pooled_gain.groupby(["model_family", "feature_set"], observed=True)
        [["pct_improvement_rmse", "pct_improvement_mae", "pct_improvement_wape"]]
        .mean()
        .round(4)
    )


## Summary and downstream use

This notebook writes `benchmark_metrics_{run_mode}.csv`, `benchmark_predictions_{run_mode}.parquet`, `benchmark_model_summary_{run_mode}.csv`, `feature_set_summary_{run_mode}.csv`, and `benchmark_gain_summary_{run_mode}.csv` under `outputs/ml_models/`. It fits one model per `(model_family, feature_set, horizon)` using fixed starting hyperparameters and the chronological holdout. It does not perform Optuna tuning or produce SHAP outputs; those steps are handled in later notebooks.
